In [1]:
# import required libraries
from kafka import KafkaConsumer, KafkaProducer
import avro.schema
import avro.io
import io
import hashlib, json
import datetime
import threading

In [2]:
def serialize(schema, obj):
    bytes_writer = io.BytesIO()
    encoder = avro.io.BinaryEncoder(bytes_writer)
    writer = avro.io.DatumWriter(schema)
    writer.write(obj, encoder)
    return bytes_writer.getvalue()

In [3]:
def deserialize(schema, raw_bytes):
    bytes_reader = io.BytesIO(raw_bytes)
    decoder = avro.io.BinaryDecoder(bytes_reader)
    reader = avro.io.DatumReader(schema)
    return reader.read(decoder)

In [4]:
schema_file = 'transaction.avsc'
txschema = avro.schema.parse(open(schema_file).read())
schema_file = 'submit.avsc'
submitschema = avro.schema.parse(open(schema_file).read())
schema_file = 'result.avsc'
resultschema = avro.schema.parse(open(schema_file).read())

In [5]:
# Connect to kafka broker running in your local host (docker). Change this to your kafka broker if needed
kafka_broker = 'lab.aimet.tech:9092'

In [6]:
producer = KafkaProducer(bootstrap_servers=[kafka_broker])

In [7]:
txconsumer = KafkaConsumer(
    'transaction',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(txschema, x))
resultconsumer = KafkaConsumer(
    'result',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(resultschema, x))

In [8]:
def gen_signature(txid, payer, payee, amount, token):
    o = {'txid': txid, 'payer': payer, 'payee': payee, 'amount': amount, 'token': token}
    return hashlib.md5(json.dumps(o, sort_keys=True).encode('utf-8')).hexdigest()

In [9]:
vid_phao = 'V720466'
token_phao = 'c9e5d9ecd2f2a3ebc9897324d31fc855'

In [10]:
def thread_listener():
    for message in resultconsumer:
        obj = message.value
        if (obj['vid'] == vid_phao):
            if obj['code'] != 200:
                print(f"message with: vid: {obj['vid']}, txid: {obj['txid']}, code: {obj['code']} is wrong")
            else:
                with open('result.txt', "a")as f:
                    f.write(f"timestamp: {obj['timestamp']}, vid: {obj['vid']}, txid: {obj['txid']}, checksum: {obj['checksum']}, code: {obj['code']}, message: {obj['message']}\n")
                print(f"timestamp: {obj['timestamp']}, vid: {obj['vid']}, txid: {obj['txid']}, checksum: {obj['checksum']}, code: {obj['code']}, message: {obj['message']}\n")
        else:
            print(f"message with: vid: {obj['vid']}, txid: {obj['txid']} isn't the same as my vid")

In [11]:
listener_thread = threading.Thread(target=thread_listener, daemon=True)
listener_thread.start()

In [12]:
for message in txconsumer:
    obj = message.value
    sig = gen_signature(obj['txid'], obj['payer'], obj['payee'], obj['amount'], token_phao)
    message_submit = {
        'vid':vid_phao,
        'txid':obj['txid'],
        'signature':sig
    }
    data_serelizer = serialize(schema=submitschema, obj=message_submit)
    producer.send('submit', data_serelizer)
    

timestamp: 1774949808, vid: V720466, txid: TX09984, checksum: 090150a6e1c8659629aae0ad3d5d246a, code: 200, message: Confirm

timestamp: 1774949817, vid: V720466, txid: TX03144, checksum: 10536405e43d01c58f0b7f233370ff52, code: 200, message: Confirm

timestamp: 1774949827, vid: V720466, txid: TX01119, checksum: 37f5b1ea67db314d902f9ddeecdc0627, code: 200, message: Confirm

timestamp: 1774949834, vid: V720466, txid: TX00923, checksum: 3f295821d070e59d4662029e6df4fa4a, code: 200, message: Confirm

message with: vid: V559635, txid: TX00923 isn't the same as my vid
timestamp: 1774949844, vid: V720466, txid: TX01123, checksum: 0ea339dbc602384f4f7faba1157c6e02, code: 200, message: Confirm

timestamp: 1774949850, vid: V720466, txid: TX01735, checksum: 2630325ab3b425bb156d1b4fc8dca215, code: 200, message: Confirm

timestamp: 1774949858, vid: V720466, txid: TX05842, checksum: cb9314b632487250caf7dc6cca193f10, code: 200, message: Confirm

timestamp: 1774949867, vid: V720466, txid: TX05323, checks

KeyboardInterrupt: 